In [3]:
import torch
print(torch.__version__, torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime -> Change runtime type -> T4 GPU")
print(torch.cuda.get_device_name(0))

2.11.0+cu128 True
Tesla T4


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
ZIP = '/content/drive/MyDrive/submission.zip'

!rm -rf /content/submission
!unzip -q "$ZIP" -d /content/
%cd /content/submission
!ls

/content/submission
colab_train.ipynb	gallery_probe.py	README.md
dataset			generate_pairs.py	requirements.txt
dataset_preparation.py	identity_manifest.json	roc_analysis.py
dataset_stats.json	make_report.py		splits.json
evaluate_pairs.py	model.py		train.py


In [6]:
import json
splits = json.load(open('splits.json'))
for k, v in splits.items():
    print(k, len(v), 'identities')

overlap = set(splits['train']) & (set(splits['val']) | set(splits['test']))
print('train/eval overlap:', len(overlap))
assert not overlap

train 1500 identities
val 60 identities
test 120 identities
train/eval overlap: 0


In [7]:
!python train.py --root . --epochs 30 --batch-p 24 --batch-k 4 --lr 1e-3 --workers 2

device=cuda
7096 images / 1500 identities, 73 batches of 96
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 138MB/s]
[  1/30] loss=20.3789 (arc=19.9237 tri=0.4551) acc=0.001 | val AUC=0.7690 EER=0.3035 | 21s
        saved (val AUC 0.7690)
[  2/30] loss=18.9517 (arc=18.5393 tri=0.4124) acc=0.005 | val AUC=0.7492 EER=0.3215 | 16s
[  3/30] loss=17.9797 (arc=17.5976 tri=0.3822) acc=0.004 | val AUC=0.8144 EER=0.2705 | 16s
        saved (val AUC 0.8144)
[  4/30] loss=15.7731 (arc=15.4393 tri=0.3338) acc=0.019 | val AUC=0.8545 EER=0.2328 | 16s
        saved (val AUC 0.8545)
[  5/30] loss=13.8849 (arc=13.5960 tri=0.2888) acc=0.037 | val AUC=0.8865 EER=0.1890 | 16s
        saved (val AUC 0.8865)
[  6/30] loss=12.3279 (arc=12.0654 tri=0.2626) acc=0.058 | val AUC=0.8746 EER=0.2107 | 16s
[  7/30] loss=10.9499 (arc=10.7199 tri=0.2300) acc=0.084 | val AUC=0.8946 EER=0.1845 | 16s
      

In [8]:
import json
h = json.load(open('checkpoints/training_history.json'))
for r in h['history']:
    print(f"epoch {r['epoch']:3d}  loss {r['loss']:.4f}  acc {r['train_acc']:.3f}  "
          f"val AUC {r['val_auc']:.4f}  val EER {r['val_eer']:.4f}")
print('best val AUC:', h['best_val_auc'])

epoch   1  loss 20.3789  acc 0.001  val AUC 0.7690  val EER 0.3035
epoch   2  loss 18.9517  acc 0.005  val AUC 0.7492  val EER 0.3215
epoch   3  loss 17.9797  acc 0.004  val AUC 0.8144  val EER 0.2705
epoch   4  loss 15.7731  acc 0.019  val AUC 0.8545  val EER 0.2328
epoch   5  loss 13.8849  acc 0.037  val AUC 0.8865  val EER 0.1890
epoch   6  loss 12.3279  acc 0.058  val AUC 0.8746  val EER 0.2107
epoch   7  loss 10.9499  acc 0.084  val AUC 0.8946  val EER 0.1845
epoch   8  loss 9.7867  acc 0.117  val AUC 0.8964  val EER 0.1858
epoch   9  loss 8.8116  acc 0.156  val AUC 0.9163  val EER 0.1633
epoch  10  loss 7.9242  acc 0.202  val AUC 0.9132  val EER 0.1630
epoch  11  loss 6.9590  acc 0.264  val AUC 0.9051  val EER 0.1735
epoch  12  loss 6.3369  acc 0.306  val AUC 0.9279  val EER 0.1510
epoch  13  loss 5.6693  acc 0.381  val AUC 0.9347  val EER 0.1445
epoch  14  loss 5.2057  acc 0.431  val AUC 0.9305  val EER 0.1460
epoch  15  loss 4.4864  acc 0.499  val AUC 0.9318  val EER 0.1467
epo

In [9]:
!cp checkpoints/best_model.pth checkpoints/training_history.json /content/drive/MyDrive/

from google.colab import files
files.download('checkpoints/best_model.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>